In [1]:
import sys
from collections import Counter
from pathlib import Path

import spacy
import unicodedataplus as ud

In [2]:
def scriptScore(token, targetScript: str = 'ARABIC'):
    """
    Returns a score from 0.0 to 1.0 representing the proportion
    of characters in 'token' belonging to 'targetScript'.
    """
    if not token:
        return 0.0

    matchCount = 0
    # Normalize target script to uppercase for comparison
    targetScript = targetScript.upper()

    ArabicVowels = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652"
    for ch in token:
        script = ud.script(ch).upper() if ch not in ArabicVowels else 'ARABIC'
        if script in (targetScript, "COMMON"):
            matchCount += 1
        else:
            print(ch, script)

    return matchCount / len(token)

In [3]:
ArabicVowels = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652"

for ch in ArabicVowels:
    print(f"'{ch}': {scriptScore(ch)} - {ud.script(ch).upper()}")

'ً': 1.0 - INHERITED
'ٌ': 1.0 - INHERITED
'ٍ': 1.0 - INHERITED
'َ': 1.0 - INHERITED
'ُ': 1.0 - INHERITED
'ِ': 1.0 - INHERITED
'ّ': 1.0 - INHERITED
'ْ': 1.0 - INHERITED


In [4]:
ud.script(" ")

'Common'

In [5]:
def tokenize(text: str):
    nlp = spacy.blank('ur')

    # Spacy based tokenization
    doc = nlp(text)
    return  doc

In [6]:
def tokens2Vocab(doc, isAlpha: bool = True, scoreThreshold: float = 0.9) -> tuple[Counter, int]:

    words = [token.text for token in doc if (isAlpha and token.is_alpha) and scriptScore(token.text) >= scoreThreshold]
    wordCount = len(words)

    vocab = Counter(words)

    return vocab, wordCount

In [7]:
urduText = """
فرض کرو ہم تارے ہوتے
ايک دوجے کو دور دور سے ديکھ ديکھ کر جلتے بجتے
اور پھر ايک دن
شاخِ فلک سے گرتے اور تاريک خلاؤں ميں کھو جاتے
دريا کے دو دھارے ہوتے،
اپنى اپنى موج ميں بہتے
اور سمندر تک اس اَندھى، وحشى اور منہ زور مسافت
کے جادو ميں تنہا رہتے
فرض کررو ہم بھور سمے کے پنچھى ہوتے،
اُڑتے اُڑتے ايک دوجے کو چھوتے اور پھر
کھلے گگن کى گہرى اور بے صرفہ آنکھوں ميں کھو جاتے
اور بہار کے جھونکے ہوتے،
موسم کے اک بےنقشہ خواب ميں ملتے
ملتے اور جدا ہو جاتے
خشک زمينوں کے ہاتھوں پر سبز لکيريں کندہ کرتے
اور ان ديکھے سپنے بوتے
اپنے اپنے رو کر چين سے سو جاتے
فرض کرو ہم جو کچھ اب ہيں وہ ناں ہوتے۔۔۔
"""

In [8]:
txt = Path(r"training_Urdu_PK.txt").read_text(encoding='utf8')
if len(txt) > 1_000_000:
    txt = txt[:1_000_000]
    print("Trimming input text size to 1,000,000")
else:
    print(f"Input text size: {len(txt):,}")

Input text size: 119,238


In [9]:
doc = tokenize(txt)
tokenCount = len(doc)
vocabulary, wordCount = tokens2Vocab(doc)
vocabCount = len(vocabulary)
print(f"Totals:: {tokenCount=:,} {wordCount=:,} {vocabCount=:,}")


Totals:: tokenCount=28,639 wordCount=25,295 vocabCount=4,041


In [10]:
print(f"{'Word':<15} | Frequency | Score")
print("-" * 25)
for w, f in vocabulary.most_common():
    s = scriptScore(w)
    if s < 1.0 :
        print(f"{w:15} | {f:9} | {s}")

print(f"\n{'Word':<15} | Frequency n(%) | Score")
print("-" * 25)
for w, f in vocabulary.most_common(n=100):
    s = scriptScore(w)
    print(f"{w:15} | {f:9}({(f/wordCount)*100:0.2f}%) | {s}")

Word            | Frequency | Score
-------------------------

Word            | Frequency n(%) | Score
-------------------------
کے              |       989(3.91%) | 1.0
کی              |       841(3.32%) | 1.0
ہے              |       829(3.28%) | 1.0
میں             |       800(3.16%) | 1.0
اور             |       728(2.88%) | 1.0
کا              |       548(2.17%) | 1.0
سے              |       547(2.16%) | 1.0
ہیں             |       384(1.52%) | 1.0
اس              |       363(1.44%) | 1.0
کو              |       285(1.13%) | 1.0
نے              |       280(1.11%) | 1.0
ان              |       262(1.04%) | 1.0
بھی             |       228(0.90%) | 1.0
کہ              |       197(0.78%) | 1.0
ایک             |       187(0.74%) | 1.0
کر              |       169(0.67%) | 1.0
پر              |       169(0.67%) | 1.0
و               |       167(0.66%) | 1.0
یہ              |       165(0.65%) | 1.0
نہیں            |       165(0.65%) | 1.0
کیا             |       163(0.64%) | 1.0
وہ       

In [26]:
Words5kTxt = Path(r"Urdu5k.txt").read_text(encoding='utf8')
lines = [line.strip() for line in Words5kTxt.splitlines()]
len(lines)

10000

In [24]:
vocab = []
i = 0
while i < len(lines):
    l1 = scriptScore(lines[i]) == 1.0
    l2 = scriptScore(lines[i+1]) == 1.0
    if l1 and l2:
        w = lines[i]
        f = lines[i+1]
        assert f.isdigit(), f"{i+1} line is not freq"
        vocab.append((w, f))
    else:
        print(lines[i], lines[i+1])
    i += 2

In [25]:
len(vocab)

5000

In [41]:
for i, l in enumerate(lines, start=1):
    if i % 2 == 0:
        # print(i, l)
        assert l.strip().isdigit(), f"{i} line is not numeric"
    else:
        assert scriptScore(l.strip()) == 1.0

In [31]:
lines[:]

['ﮐﮯ',
 '743949',
 'ﻣﻴﮟ',
 '582882',
 'ﮐﯽ',
 '575545',
 'ﮨﮯ',
 '466908',
 'اور',
 '413788']

In [80]:
CORRECT_URDU_CHARACTERS: dict = {'آ': ['ﺁ', 'ﺂ'],
                                 'أ': ['ﺃ'],
                                 'ا': ['ﺍ', 'ﺎ', ],
                                 'ب': ['ﺏ', 'ﺐ', 'ﺑ', 'ﺒ'],
                                 'پ': ['ﭖ', 'ﭗ', 'ﭘ', 'ﭙ'],
                                 'ت': ['ﺕ', 'ﺖ', 'ﺗ', 'ﺘ'],
                                 'ٹ': ['ﭦ', 'ﭧ', 'ﭨ', 'ﭩ'],
                                 'ث': ['ﺛ', 'ﺜ', 'ﺚ'],
                                 'ج': ['ﺝ', 'ﺞ', 'ﺟ', 'ﺠ'],
                                 'ح': ['ﺡ', 'ﺣ', 'ﺤ', 'ﺢ'],
                                 'خ': ['ﺧ', 'ﺨ', 'ﺦ'],
                                 'د': ['ﺩ', 'ﺪ'],
                                 'ذ': ['ﺬ', 'ﺫ'],
                                 'ر': ['ﺭ', 'ﺮ'],
                                 'ز': ['ﺯ', 'ﺰ', ],
                                 'س': ['ﺱ', 'ﺲ', 'ﺳ', 'ﺴ', ],
                                 'ش': ['ﺵ', 'ﺶ', 'ﺷ', 'ﺸ'],
                                 'ص': ['ﺹ', 'ﺺ', 'ﺻ', 'ﺼ', ],
                                 'ض': ['ﺽ', 'ﺾ', 'ﺿ', 'ﻀ'],
                                 'ط': ['ﻃ', 'ﻄ', 'ﻂ'],
                                 'ظ': ['ﻅ', 'ﻇ', 'ﻆ', 'ﻈ'],
                                 'ع': ['ﻉ', 'ﻊ', 'ﻋ', 'ﻌ', ],
                                 'غ': ['ﻍ', 'ﻏ', 'ﻐ', 'ﻎ'],
                                 'ف': ['ﻑ', 'ﻒ', 'ﻓ', 'ﻔ', ],
                                 'ق': ['ﻕ', 'ﻖ', 'ﻗ', 'ﻘ', ],
                                 'ل': ['ﻝ', 'ﻞ', 'ﻟ', 'ﻠ', ],
                                 'م': ['ﻡ', 'ﻢ', 'ﻣ', 'ﻤ', ],
                                 'ن': ['ﻥ', 'ﻦ', 'ﻧ', 'ﻨ', ],
                                 'چ': ['ﭺ', 'ﭻ', 'ﭼ', 'ﭽ'],
                                 'ڈ': ['ﮈ', 'ﮉ'],
                                 'ڑ': ['ﮍ', 'ﮌ'],
                                 'ژ': ['ﮋ', ],
                                 'ک': ['ﮎ', 'ﮏ', 'ﮐ', 'ﮑ', 'ﻛ', 'ك'],
                                 'گ': ['ﮒ', 'ﮓ', 'ﮔ', 'ﮕ'],
                                 'ں': ['ﮞ', 'ﮟ'],
                                 'و': ['ﻮ', 'ﻭ', 'ﻮ', ],
                                 'ؤ': ['ﺅ'],
                                 'ھ': ['ﮪ', 'ﮬ', 'ﮭ', 'ﻬ', 'ﻫ', 'ﮫ'],
                                 'ہ': ['ﻩ', 'ﮦ', 'ﻪ', 'ﮧ', 'ﮩ', 'ﮨ', 'ه', ],
                                 'ۂ': [],
                                 'ۃ': ['ة'],
                                 'ء': ['ﺀ'],
                                 'ی': ['ﯼ', 'ى', 'ﯽ', 'ﻰ', 'ﻱ', 'ﻲ', 'ﯾ', 'ﯿ', 'ي', 'ﻳ', 'ﻴ'],
                                 'ئ': ['ﺋ', 'ﺌ', ],
                                 'ے': ['ﮮ', 'ﮯ' ],
                                 'ۓ': [],
                                 '۰': ['٠'],
                                 '۱': ['١'],
                                 '۲': ['٢'],
                                 '۳': ['٣'],
                                 '۴': ['٤'],
                                 '۵': ['٥'],
                                 '۶': ['٦'],
                                 '۷': ['٧'],
                                 '۸': ['٨'],
                                 '۹': ['٩'],
                                 '۔': [],
                                 '؟': [],
                                 '٫': [],
                                 '،': [],
                                 'لا': ['ﻻ', 'ﻼ', 'ﻵ'],
                                 '': ['ـ']

                                 }

_TRANSLATOR = {}
for key, value in CORRECT_URDU_CHARACTERS.items():
    _TRANSLATOR.update(dict.fromkeys(map(ord, value), key))

In [17]:
def getUniRep(txt):
    assert txt is not None
    assert type(txt) == str

    uc = [f"U+{ord(c):04X}" for c in txt]
    return " ".join(uc[::-1])

In [74]:
def isArabicBlock(text):
    """Checks if all characters are within the main Arabic Unicode block."""
    return all('\u0600' <= char <= '\u06FF' for char in text if not char.isspace())

In [82]:
with open(r"Urdu5k.sym", "w", encoding="utf-8") as dic:
    for i, (w, f) in enumerate(vocab):
        ww = w.translate(_TRANSLATOR)
        if not isArabicBlock(ww):
            print(i, getUniRep(w), w)
            ww = w.translate(_TRANSLATOR)
            print(i, getUniRep(ww), ww, '\n')
        dic.write(f"{ww}${f}\n")

In [68]:
w, f = vocab[1427]
print(getUniRep(w), w)
ww = w.translate(_TRANSLATOR)
print(getUniRep(ww), ww)

U+06BA U+FEEE U+FEE4 U+FEF4 U+FEC8 U+FEE8 U+FE97 ﺗﻨﻈﻴﻤﻮں
U+06BA U+0648 U+0645 U+06CC U+0638 U+0646 U+062A تنظیموں
